# Experiment Dust emission 
In this exercise you will learn:
* how to emit dust
* how to activate aerosol-radiation interaction

### Objective:
- Learn how to emit dust and how to activate aerosol-radiation-interaction (ARI)

### Simulation Design:
- Run two global simulations (with and without ARI)

### Analysis:
- Plot and understand the role of dust on surface short-wave and longwave radiation 




## 1. Prepare the emission

Open the aerosol emission xml (Ex02\_aeroemiss.xml) and insert the correct numbers from the table below. (<button data-commandlinker-command="terminal:create-new">Create Terminal</button>).

 | dust modes | d\_g0 | d\_g3 | sigma\_g | rho |
 | ----- | ----- | ------ | ----- | ----- |
 | dusta | 6.445E-7 | 1.500E-6 | 1.7E+0 | 2.650E+3 |
 | dustb | 3.454E-6 | 6.700E-6 | 1.6E+0 | 2.650E+3 |
 | dustc | 8.672E-6 | 1.420E-5 | 1.5E+0 | 2.650E+3 |

## 2. Input data
Insert the correct paths to link the input data

In [ ]:
# Output directory
export OUTDIR=/path/to/scratch/

# Path to the xml files
export XMLDIR=/path/to/xmlfiles/

# input data
export INDIR=/path/to/input-data

# ICON-ART code directory
export ICONDIR=path/to/icon-model

In [ ]:
if [ ! -d $OUTDIR ]; then
    mkdir -p $OUTDIR
fi

cd $OUTDIR

# copy icon binary to OUTDIR
cp ${ICONDIR}/bin/icon icon.exe

In [ ]:
# Link input files in experiment directory
ln -sf ${INDIR}/iconR2B05_DOM01.nc iconR2B05_DOM01.nc
ln -sf ${INDIR}/ExtPar_R2B5_DOM1.nc extpar_iconR2B05_DOM01.nc
ln -sf ${INDIR}/ifs2icon_R2B05_2014032900.nc ifs2icon_R2B05_DOM01.nc
# Link files needed for physics
ln -sf ${ICONDIR}/data/ECHAM6_CldOptProps.nc ECHAM6_CldOptProps.nc
ln -sf ${ICONDIR}/data/rrtmg_lw.nc rrtmg_lw.nc
# Link files needed for art
ln -sf ${INDIR}/ART_STY_R2B5_DOM01_TCNR.nc ${OUTDIR}/ART_STY_iconR2B05-grid_TCNR.nc
ln -sf ${INDIR}/ART_IAE_R2B5_DOM01_TCNR.nc ${OUTDIR}/ART_IAE_iconR2B05-grid_TCNR.nc


## 3. The runscript
The current setting neglect ARI. For the second experiment, you need to activate ARI. Make sure that you save the output in a different directory than the output for the simulation without ARI. Otherwise, the output data will be overwritten.

In [ ]:
cat > icon_master.namelist << EOF
&master_nml
 lRestart               = .false.
/
&master_time_control_nml
 experimentStartDate = "2014-03-15T00:00:00Z"
 forecastLeadTime = "P1D"
/
&master_model_nml
  model_type=1
  model_name="ATMO"
  model_namelist_filename=NAMELIST_icon-art-dust
  model_min_rank=1
  model_max_rank=65536
  model_inc_rank=1
/
&time_nml
 ini_datetime_string = "2014-03-15T00:00:00Z"
/
EOF

In [ ]:
cat > NAMELIST_icon-art-dust << EOF
&parallel_nml
 nproma         = 8  ! optimal setting 8 for CRAY; use 16 or 24 for IBM
 p_test_run     = .false.
 l_test_openmp  = .false.
 l_log_checks   = .false.
 num_io_procs   = 1   ! up to one PE per output stream is possible
 iorder_sendrecv = 3  ! best value for CRAY (slightly faster than option 1)
/
&grid_nml
 dynamics_grid_filename  = 'iconR2B05_DOM01.nc'
 dynamics_parent_grid_id = 0
 lredgrid_phys           = .false.
 lfeedback               = .true.
 ifeedback_type          = 2
/
&initicon_nml
 init_mode   = 2           ! operation mode 2: IFS
 zpbl1       = 500. 
 zpbl2       = 1000. 
! l_sst_in    = .true.
/
&run_nml
 num_lev        = 90
 lvert_nest     = .true.       ! use vertical nesting if a nest is active
 nsteps         = 240    ! 50 ! 1200 ! 7200 !
 dtime          = 360     ! timestep in seconds
 ldynamics      = .TRUE.       ! dynamics
 ltransport     = .true.
 iforcing       = 3            ! NWP forcing
 ltestcase      = .false.      ! false: run with real data
 msg_level      = 7            ! print maximum wind speeds every 5 time steps
 ltimer         = .false.      ! set .TRUE. for timer output
 timers_level   = 1            ! can be increased up to 10 for detailed timer output
 output         = "nml"
 lart           = .true.
/
&nwp_phy_nml
 inwp_gscp       = 1
 inwp_convection = 1
 inwp_radiation  = 4
 inwp_cldcover   = 1
 inwp_turb       = 1
 inwp_satad      = 1
 inwp_sso        = 1
 inwp_gwd        = 1
 inwp_surface    = 1
 icapdcycl       = 3 ! apply CAPE modification to improve diurnalcycle over tropical land (optimizes NWP scores)
 latm_above_top  = .false. ! the second entry refers to the nested domain (if present)
 efdt_min_raylfric = 7200.
 itype_z0         = 2
 icpl_aero_conv   = 1
 icpl_aero_gscp   = 1
 icpl_o3_tp       = 1
 ! resolution-dependent settings - please choose the appropriate one
 dt_rad    = 2160.
 dt_conv   = 720.
 dt_sso    = 1440.
 dt_gwd    = 1440.
/
&nwp_tuning_nml
 itune_albedo  = 1
 tune_gkwake   = 1.5
 tune_gfrcrit  = 0.425
 tune_grcrit   = 0.5
 tune_gkdrag   = 0.16
 tune_zvz0i    = 0.85
 tune_box_liq_asy = 3.25
 tune_minsnowfrac = 0.2
 tune_gfluxlaun  = 3.75e-3
 tune_rcucov = 0.075
 tune_rhebc_land = 0.825
/
&turbdiff_nml
 tkhmin       = 0.6
 tkhmin_strat = 1.0
 tkmmin       = 0.75
 tkmmin_strat = 1.5
 pat_len      =  750.
 c_diff       =  0.2
 rat_sea      =  0.8
 rlam_heat    = 10.
 ltkesso      = .true.
 frcsmot      = 0.2
 imode_frcsmot = 2
 icldm_turb   = 1
 itype_sher   = 1
 ltkeshs      = .true.
 a_hshr       = 2.0
/
&lnd_nml
 ntiles         = 3
 nlev_snow      = 1
 lmulti_snow    = .false.
 itype_heatcond = 3
 idiag_snowfrac = 20
 itype_snowevap = 2
 itype_canopy   = 2
 lsnowtile      = .true.
 lseaice        = .true.
 llake          = .false.
 itype_lndtbl   = 4
 itype_evsl     = 4
 itype_root     = 2
 cwimax_ml      = 5.e-4
 c_soil         = 1.25
 c_soil_urb     = 0.5
! sstice_mode    = 2
 lprog_albsi    = .true.
/
&radiation_nml
 irad_o3       = 7
 irad_aero     = 6 ! set irad_aero=6 if iart_ari=0 --> aerosol optical properties are taken from climatology;
                   ! set irad_aero=9 if iart_ari=1 --> optical properties depend on aerosol concentrations
 albedo_type   = 2 ! Modis albedo
 vmr_co2       = 390.e-06 ! values representative for 2012
 vmr_ch4       = 1800.e-09
 vmr_n2o       = 322.0e-09
 vmr_o2        = 0.20946
 vmr_cfc11     = 240.e-12
 vmr_cfc12     = 532.e-12
 ecrad_data_path='${ICONDIR}/externals/ecrad/data'
/
&nonhydrostatic_nml
 itime_scheme   = 4
 vwind_offctr   = 0.2
 damp_height    = 50000.
 rayleigh_coeff = 0.1
 divdamp_order  = 24    ! for data assimilation runs, '2' provides extra-strong filtering of gravity waves
 divdamp_type   = 32    !!! optional: 2 for assimilation cycle if very strong gravity-wave filtering is desired
 divdamp_fac    = 0.004
 igradp_method  = 3
 l_zdiffu_t     = .true.
 thslp_zdiffu   = 0.02
 thhgtd_zdiffu  = 125.
 htop_moist_proc= 22000.
 hbot_qvsubstep = 19000. ! use 19000. with R3B7
 htop_aero_proc  = 22000.
/
&sleve_nml
 min_lay_thckn   = 20.
 max_lay_thckn   = 400.   ! maximum layer thickness below htop_thcknlimit
 htop_thcknlimit = 14000. ! this implies that the upcoming COSMO-EU nest will have 60 levels
 top_height      = 75000.
 stretch_fac     = 0.9
 decay_scale_1   = 4000.
 decay_scale_2   = 2500.
 decay_exp       = 1.2
 flat_height     = 16000.
/
&dynamics_nml
 divavg_cntrwgt = 0.50
 lcoriolis      = .true.
/
&transport_nml
 ctracer_list  = '12345'
 ivadv_tracer  = 3,3,3,3,3
 itype_hlimit  = 3,4,4,4,4
 ihadv_tracer  = 52,2,2,2,2
 iadv_tke      = 0
/
&diffusion_nml
 hdiff_order      = 5
 itype_vn_diffu   = 1
 itype_t_diffu    = 2
 hdiff_efdt_ratio = 24.0
 hdiff_smag_fac   = 0.025
 lhdiff_vn        = .TRUE.
 lhdiff_temp      = .TRUE.
/
&interpol_nml
 nudge_zone_width  = 8
 lsq_high_ord      = 3
 l_intp_c2l        = .true.
 l_mono_c2l        = .true.
/
&extpar_nml
 itopo          = 1
 n_iter_smooth_topo = 1
 heightdiff_threshold = 3000.
/
&io_nml
 itype_pres_msl = 4  ! IFS method with bug fix for self-consistency between SLP and geopotential
 itype_rh       = 1  ! RH w.r.t. water
 restart_write_mode = 'joint procs multifile'
/
&output_nml
 filetype                     = 4                        ! output format: 2=GRIB2, 4=NETCDFv2
 dom                          = -1                        ! write all domains
 output_bounds                = 0., 86400., 3600.  ! start, end, increment
 steps_per_file               = 1
 include_last                 = .TRUE.
 output_filename              = 'icon-art-dust-remap'                ! file name base
 ml_varlist                   = 'group:ART_AEROSOL' ,'z_mc','temp','z_ifc','rho','emiss_dusta','emiss_dustb', 'emiss_dustc'
 remap                        = 1
 reg_lon_def                  = -180.,0.5,179.5
 reg_lat_def                  = 90.,-0.5, -90.
/
&art_nml
 lart_aerosol    = .TRUE.
 lart_diag_out   = .TRUE.
 iart_dust       = 1
 iart_ari        = 0         ! iart_ari = 0 --> no aerosol-radiation interaction; 
                             ! iart_ari = 1 --> consider aerosol-radiation interaction
 iart_init_aero  = 5
 cart_io_suffix    = 'TCNR'
 cart_aerosol_xml  = '${XMLDIR}/Ex02_aerotracer.xml'
 cart_modes_xml    = '${XMLDIR}/Ex02_modes.xml'
 cart_aero_emiss_xml   = '${XMLDIR}/Ex02_aeroemiss.xml'
 cart_diagnostics_xml = '${XMLDIR}/Ex02_diagnostics.xml'
 cart_input_folder    = '${OUTDIR}/'
/
EOF

## 4. Prepare the ICON-ART batch job on Levante
Change to your project account

In [ ]:
cat > ${OUTDIR}/job_ICON << ENDFILE
#!/bin/bash
#SBATCH --job-name=EXP02_ART            # Specify job name
#SBATCH --partition=compute             # Specify partition name
#SBATCH --nodes=2                       # Specify number of nodes
#SBATCH --ntasks-per-node=128           # Specify number of (MPI) tasks on each node
#SBATCH --time=01:00:00                 # Set a limit on the total run time
#SBATCH --account=????                # Charge resources on this project account

unset SLURM_EXPORT_ENV 
unset SLURM_MEM_PER_NODE
unset SBATCH_EXPORT

export LD_LIBRARY_PATH="/usr/lib:/usr/lib64:/sw/spack-levante/netcdf-c-4.8.1-2k3cmu/lib:/sw/spack-levante/netcdf-fortran-4.5.3-k6xq5g/lib:/sw/spack-levante/hdf5-1.12.1-tvymb5/lib:/sw/spack-levante/eccodes-2.21.0-3ehkbb/lib64:/sw/spack-levante/intel-oneapi-mkl-2022.0.1-ttdktf/mkl/2022.0.1/lib/intel64/"

export OMPI_MCA_pml="ucx"
export OMPI_MCA_btl=self
export OMPI_MCA_osc="pt2pt"
export UCX_IB_ADDR_TYPE=ib_global

export OMPI_MCA_coll="^ml,hcoll"
export OMPI_MCA_coll_hcoll_enable="0"
export HCOLL_ENABLE_MCAST_ALL="0"
export HCOLL_MAIN_IB=mlx5_0:1
export UCX_NET_DEVICES=mlx5_0:1
export UCX_TLS=mm,knem,cma,dc_mlx5,dc_x,self
export UCX_UNIFIED_MODE=y
export HDF5_USE_FILE_LOCKING=FALSE
export OMPI_MCA_io="romio321"
export UCX_HANDLE_ERRORS=bt


ulimit -s 102400
ulimit -c 0

srun -l --cpu_bind=cores --distribution=block:cyclic --propagate=STACK,CORE ./icon.exe


ENDFILE


## 5. Submit the job

In [ ]:
chmod +x job_ICON
sbatch job_ICON

In [ ]:
squeue -u $USER

# 6. Re-run the simulation with Aerosol-Radiation Interaction and compare the two simulations
1. re-name OUTDIR
2. activate aerosol-radiation interaction (change namelist setting of irad_aero and iart_ari)
3. submit the job
4. run the plotting script ([Plot_ex2.ipynb](Plot_ex2.ipynb)) to compare the simulations